# M05 - Data engineering: audits, duplicates, labels, and cards

**Hardware:** CPU (seconds) · **Time:** 3-5 hours ·
**Prerequisite:** M04, basic pandas

A model can only be as good as the data it is trained on, and most data
problems are visible before any model is trained: missing fields, duplicated
rows, inconsistent labels, near-duplicates across splits, and class imbalance.
This notebook audits a deliberately messy support-ticket dataset, quantifies
two of those problems, and produces a dataset card.

The classifier used here is a small TF-IDF nearest-centroid model implemented
in the notebook. It is not the point: it exists so that changes in the data
produce a measurable change in a score. No scikit-learn dependency is needed.

**You will produce:** a cleaning report, a leakage audit, a label-noise curve,
an inter-annotator agreement figure, and `outputs/m05_dataset_card.md`.

## 1. Setup and the raw data

`scripts/make_training_data.py` generates two ticket files from a fixed seed:

* `tickets_clean.csv` - 600 rows, three balanced labels, unique text.
* `tickets_messy.csv` - the same rows plus 40 exact duplicates, 10 blank texts,
  and label typos (`Tech `, `acct`, `Biling`).

The mess is intentional and documented, which is the first lesson: a dataset
with known defects is easier to trust than one whose defects are unknown.

In [1]:
from pathlib import Path
import hashlib
import re
import sys
from collections import Counter

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "projects").is_dir())
sys.path.insert(0, str(ROOT / "projects" / "llm-fine-tuning"))

import llm_core as lc

lc.ensure_datasets(quiet=True)
clean = pd.DataFrame(lc.load_tickets(messy=False))
messy = pd.DataFrame(lc.load_tickets(messy=True))
print("clean rows:", len(clean), "| messy rows:", len(messy))
print("columns:", list(messy.columns))
print(messy.head(3).to_string(max_colwidth=60))
print("\nmessy label values:", messy["label"].value_counts(dropna=False).to_dict())

clean rows: 600 | messy rows: 650
columns: ['ticket_id', 'customer_id', 'created_at', 'text', 'label', 'resolution_note']
  ticket_id customer_id  created_at                                                        text    label                                  resolution_note
0     T1184        C030  2026-01-21  I cannot reset my password for the account on order 50030.  account  Password reset completed and profile corrected.
1     T1522        C087  2026-01-18       Please send a receipt for order 42497 for my records.  billing   Refund approved and payment adjustment posted.
2     T1103        C017  2026-03-15           My invoice for order 82700 shows the wrong total.  billing   Refund approved and payment adjustment posted.

messy label values: {'account': 214, 'billing': 213, 'technical': 211, 'Tech ': 4, 'acct': 4, 'Biling': 4}


## 2. Audit before cleaning

Every audit answers one question with a number. The four below are the minimum
set for tabular text data:

1. How many rows are missing the field the model uses?
2. How many rows are exact duplicates?
3. Which label spellings exist, and which are the same label?
4. Is the class balance acceptable?

A cleaning script that does not report these before/after numbers cannot be
reviewed, because the reviewer cannot tell what it changed.

In [2]:
def audit(frame, name):
    text = frame["text"].fillna("").astype(str).str.strip()
    return {
        "dataset": name,
        "rows": len(frame),
        "blank_text": int((text == "").sum()),
        "duplicate_text": int(text.duplicated().sum()),
        "labels": frame["label"].nunique(dropna=False),
        "top_label_share": round(frame["label"].value_counts(normalize=True).iloc[0], 3),
    }


print(pd.DataFrame([audit(messy, "messy"), audit(clean, "clean")]).to_string(index=False))

dataset  rows  blank_text  duplicate_text  labels  top_label_share
  messy   650          10              49       6            0.329
  clean   600           0               0       3            0.333


## 3. Normalise labels and remove duplicates

Label normalisation is a mapping, not a guess: `Tech ` and `technical` are the
same class, `acct` and `account` are the same class, `Biling` and `billing` are
the same class. Write the mapping down and test it. Dropping blank rows and
exact duplicates is safe here because the generator tells us duplicates are
resubmissions, not independent observations.

Order matters: normalise labels *before* deduplicating, so two rows that differ
only in label spelling still collapse.

In [3]:
LABEL_MAP = {"tech": "technical", "technical": "technical",
             "acct": "account", "account": "account",
             "biling": "billing", "billing": "billing"}


def clean_tickets(frame, dedupe=True):
    out = frame.copy()
    out["text"] = out["text"].fillna("").astype(str).str.strip()
    out["label_raw"] = out["label"]
    out["label"] = out["label"].astype(str).str.strip().str.lower().map(LABEL_MAP)
    unknown = out["label"].isna()
    out = out[~unknown]
    out = out[out["text"] != ""]
    if dedupe:
        out = out.drop_duplicates(subset=["text"])
    return out.reset_index(drop=True), int(unknown.sum())


audited, unknown = clean_tickets(messy, dedupe=False)
cleaned = audited.drop_duplicates(subset=["text"]).reset_index(drop=True)
print("rows dropped for unknown label:", unknown)
print("audited rows (duplicates kept):", len(audited))
print("after cleaning:", len(cleaned), "rows")
print("class balance:", cleaned["label"].value_counts().to_dict())
print("remaining duplicates:", int(cleaned["text"].duplicated().sum()))
assert len(cleaned) == 600
assert cleaned["text"].is_unique

rows dropped for unknown label: 0
audited rows (duplicates kept): 640
after cleaning: 600 rows
class balance: {'account': 200, 'billing': 200, 'technical': 200}
remaining duplicates: 0


## 4. A classifier small enough to inspect

The evaluation below needs a model that is fast, deterministic, and simple
enough that a change in accuracy can only come from the data. Multinomial
Naive Bayes fits that description: count words per class, smooth the counts,
and pick the class with the highest log-likelihood.

It is a weak model by design. If a data defect moves this model's score, the
defect is real and not an artefact of a complex optimiser.

In [4]:
TOKEN = re.compile(r"[a-z0-9']+")


def tokenize(text):
    return TOKEN.findall(text.lower())


class NaiveBayes:
    """Multinomial Naive Bayes with Laplace smoothing, written out in full."""

    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, texts, labels):
        documents = [tokenize(text) for text in texts]
        vocabulary = sorted({word for document in documents for word in document})
        self.index = {word: position for position, word in enumerate(vocabulary)}
        self.classes = sorted(set(labels))
        counts = {label: np.full(len(vocabulary), self.alpha) for label in self.classes}
        for document, label in zip(documents, labels):
            for word, count in Counter(document).items():
                counts[label][self.index[word]] += count
        self.log_likelihood = {
            label: np.log(values / values.sum()) for label, values in counts.items()
        }
        totals = Counter(labels)
        self.log_prior = {label: np.log(totals[label] / len(labels)) for label in self.classes}
        return self

    def predict(self, texts):
        predictions = []
        for text in texts:
            words = Counter(tokenize(text))
            scores = {}
            for label in self.classes:
                score = self.log_prior[label]
                for word, count in words.items():
                    position = self.index.get(word)
                    if position is not None:
                        score += count * self.log_likelihood[label][position]
                scores[label] = score
            predictions.append(max(scores, key=scores.get))
        return predictions


def accuracy(predictions, truth):
    return float(np.mean(np.array(predictions) == np.array(truth)))


rng = np.random.default_rng(0)
order = rng.permutation(len(cleaned))
cut = int(len(cleaned) * 0.7)
train_idx, test_idx = order[:cut], order[cut:]
model = NaiveBayes().fit(cleaned["text"].iloc[train_idx], cleaned["label"].iloc[train_idx])
predictions = model.predict(cleaned["text"].iloc[test_idx])
print("vocabulary:", len(model.index), "| test rows:", len(test_idx))
print("random-split accuracy:", round(accuracy(predictions, cleaned["label"].iloc[test_idx]), 3))

vocabulary: 533 | test rows: 180
random-split accuracy: 0.983


## 5. Leakage is a property of the split, not the model

The random split above can place two tickets from the same customer, or two
copies of the same text, on opposite sides. A customer-level split keeps every
ticket from a customer together. Compare the overlap and the score.

In this synthetic corpus the ticket templates are so distinctive that both
splits score highly; the structural overlap is the finding, not a dramatic
accuracy gap. The image notebook (M04) shows the same audit with a 0.244
accuracy gap, because there the near-duplicates are pixel variants rather than
identical strings.

In [5]:
customers = audited["customer_id"].unique()
rng = np.random.default_rng(1)
shuffled = rng.permutation(customers)
group_cut = int(len(shuffled) * 0.7)
group_train_customers = set(shuffled[:group_cut])
group_train = audited[audited["customer_id"].isin(group_train_customers)]
group_test = audited[~audited["customer_id"].isin(group_train_customers)]

audit_order = np.random.default_rng(9).permutation(len(audited))
audit_cut = int(len(audited) * 0.7)
audit_train_idx, audit_test_idx = audit_order[:audit_cut], audit_order[audit_cut:]
random_train_texts = set(audited["text"].iloc[audit_train_idx])
group_train_texts = set(group_train["text"])
random_overlap = int(audited["text"].iloc[audit_test_idx].isin(random_train_texts).sum())
print("random split: test rows also in train:", random_overlap)
group_overlap = int(group_test["text"].isin(group_train_texts).sum())
print("group split : test rows also in train:", group_overlap)

group_model = NaiveBayes().fit(group_train["text"], group_train["label"])
group_predictions = group_model.predict(group_test["text"])
print("group-split accuracy:", round(accuracy(group_predictions, group_test["label"]), 3))
print("customers: train", group_train["customer_id"].nunique(), "| test", group_test["customer_id"].nunique())

random split: test rows also in train: 16
group split : test rows also in train: 0
group-split accuracy: 0.984
customers: train 70 | test 30


## 6. Near-duplicates are not exact duplicates

Exact matching misses paraphrases and small edits. Jaccard similarity over word
sets catches them. A pair above 0.8 is suspicious; above 0.95 it is effectively
the same document. Report the count and inspect a few pairs before deleting
anything.

In [6]:
def jaccard(left, right):
    a, b = set(tokenize(left)), set(tokenize(right))
    return len(a & b) / max(1, len(a | b))


texts = cleaned["text"].tolist()
near_pairs = []
for i in range(len(texts)):
    for j in range(i + 1, len(texts)):
        score = jaccard(texts[i], texts[j])
        if score >= 0.8:
            near_pairs.append((i, j, round(score, 3)))
print("near-duplicate pairs (Jaccard >= 0.8):", len(near_pairs))
random_overlap = int(audited["text"].iloc[audit_test_idx].isin(random_train_texts).sum())
group_overlap = int(group_test["text"].isin(group_train_texts).sum())
for i, j, score in near_pairs[:3]:
    print(f"  {score}: {texts[i][:70]!r} <> {texts[j][:70]!r}")

near-duplicate pairs (Jaccard >= 0.8): 24976
  0.833: 'I cannot reset my password for the account on order 50030.' <> 'I cannot reset my password for the account on order 94268.'
  0.833: 'I cannot reset my password for the account on order 50030.' <> 'I cannot reset my password for the account on order 94489.'
  0.833: 'I cannot reset my password for the account on order 50030.' <> 'I cannot reset my password for the account on order 68255.'


## 7. Label noise has a measurable cost

Take the sentiment reviews, split them 70/30 **within the same template pool**,
then flip a fraction of the training labels at random (test labels stay
correct). Accuracy on the clean test set falls as noise rises. This is the
number to quote when someone proposes "just label more data quickly".

The split stays inside one template pool on purpose. Evaluating on unseen
templates is a different and much harder question, measured in M07 with the FAQ
data; here we isolate label quality from phrasing generalisation.

The flip is seeded, so the curve is reproducible. A single seed is still one
sample; the shape matters more than the exact values.

In [7]:
sentiment = pd.read_csv(lc.ensure_datasets() / "sentiment_train.csv")
rng = np.random.default_rng(0)
order = rng.permutation(len(sentiment))
cut = int(len(sentiment) * 0.7)
train_part, test_part = sentiment.iloc[order[:cut]], sentiment.iloc[order[cut:]]

noise_rows = []
for noise in (0.0, 0.2, 0.4, 0.6):
    rng = np.random.default_rng(42)
    noisy = train_part["label"].tolist()
    flip = rng.random(len(noisy)) < noise
    for index in np.where(flip)[0]:
        noisy[index] = "positive" if noisy[index] == "negative" else "negative"
    model = NaiveBayes().fit(train_part["text"], noisy)
    predictions = model.predict(test_part["text"])
    noise_rows.append({"label_noise": noise, "test_accuracy": round(accuracy(predictions, test_part["label"]), 3)})

print(pd.DataFrame(noise_rows).to_string(index=False))
assert noise_rows[-1]["test_accuracy"] < noise_rows[0]["test_accuracy"]

 label_noise  test_accuracy
         0.0          1.000
         0.2          0.868
         0.4          0.794
         0.6          0.382


## 8. Inter-annotator agreement

Raw agreement is misleading when one class dominates: two annotators who always
say "positive" agree 90% of the time on a 90% positive dataset while measuring
nothing. Cohen's kappa corrects for chance agreement.

```text
kappa = (observed agreement - chance agreement) / (1 - chance agreement)
```

A common reading: below 0.4 is weak, 0.4-0.6 moderate, 0.6-0.8 substantial,
above 0.8 strong. These are conventions, not laws.

In [8]:
def cohen_kappa(first, second):
    first, second = list(first), list(second)
    labels = sorted(set(first) | set(second))
    observed = np.mean([a == b for a, b in zip(first, second)])
    chance = sum((first.count(label) / len(first)) * (second.count(label) / len(second)) for label in labels)
    return (observed - chance) / (1 - chance)


truth = sentiment["label"].tolist()
for flip_rate in (0.05, 0.12, 0.25):
    rng = np.random.default_rng(int(flip_rate * 100))
    second = [label if rng.random() > flip_rate else ("positive" if label == "negative" else "negative") for label in truth]
    raw = float(np.mean([a == b for a, b in zip(truth, second)]))
    print(f"flip {flip_rate:.2f}: raw agreement {raw:.3f} | kappa {cohen_kappa(truth, second):.3f}")

flip 0.05: raw agreement 0.933 | kappa 0.866
flip 0.12: raw agreement 0.862 | kappa 0.723
flip 0.25: raw agreement 0.754 | kappa 0.509


## 9. The dataset card

A dataset card is the contract between the people who built the data and the
people who will train on it. It records provenance, licence, composition,
preprocessing, intended use, and limitations. Every number in it must come from
the audit above - a card with hand-written claims is worse than no card.

The digest makes the card versionable: if the CSV changes, the digest changes,
and a reviewer can tell that the card no longer describes the file.

In [9]:
digest = hashlib.sha256((lc.ensure_datasets() / "tickets_messy.csv").read_bytes()).hexdigest()
card = f"""# Dataset card: Northwind support tickets

## Dataset details
- Version: 1.0
- Format: CSV, {len(messy.columns)} columns, {len(messy)} raw rows
- Digest: `{digest}`

## Provenance
Synthetic tickets generated by `scripts/make_training_data.py` (seed 2026). No personal data, no scraped text.

## Composition
{len(messy)} raw rows; {len(cleaned)} rows after removing blanks, duplicates, and label typos.
Classes: {", ".join(f"{name} ({count})" for name, count in cleaned["label"].value_counts().sort_index().items())}.

## Collection process
Generated, not collected. Duplicates simulate resubmission; blanks simulate ingestion failure; label typos simulate free-text entry.

## Preprocessing
Trim whitespace, lowercase and map label spellings, drop blank text, drop exact duplicate text. Near-duplicates are reported, not removed.

## Uses
Teaching data engineering: auditing, cleaning, leakage analysis, label-noise experiments. Not suitable for training a production classifier.

## Limitations
- Template-generated language; models learn template cues, not intent.
- Only three classes and 600 rows.
- Near-duplicate rate is artificial.

## Licence
CC0-1.0.
"""
output_dir = ROOT / "notebooks" / "model-training" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)
card_path = output_dir / "m05_dataset_card.md"
card_path.write_text(card, encoding="utf-8")

required = ["Dataset details", "Provenance", "Composition", "Collection process",
            "Preprocessing", "Uses", "Limitations", "Licence"]
headings = {line[3:].strip() for line in card.splitlines() if line.startswith("## ")}
print("missing sections:", [section for section in required if section not in headings])
print(card[:900])
assert not [section for section in required if section not in headings]

missing sections: []
# Dataset card: Northwind support tickets

## Dataset details
- Version: 1.0
- Format: CSV, 6 columns, 650 raw rows
- Digest: `24b6521b6b7c989dbfb2d13b97cbe408e3bfce0d21b353c2b6e3c53619f90b69`

## Provenance
Synthetic tickets generated by `scripts/make_training_data.py` (seed 2026). No personal data, no scraped text.

## Composition
650 raw rows; 600 rows after removing blanks, duplicates, and label typos.
Classes: account (200), billing (200), technical (200).

## Collection process
Generated, not collected. Duplicates simulate resubmission; blanks simulate ingestion failure; label typos simulate free-text entry.

## Preprocessing
Trim whitespace, lowercase and map label spellings, drop blank text, drop exact duplicate text. Near-duplicates are reported, not removed.

## Uses
Teaching data engineering: auditing, cleaning, leakage analysis, label-noise experiments. Not suitable for train


## 10. Validation and exercises

**Exercises**

1. Lower the Jaccard threshold to 0.6. How many pairs appear? Inspect five and
   decide which are true duplicates and which are the same topic.
2. Change the label-noise experiment to 20% noise but correct half of the noisy
   labels using a rule. Does accuracy recover? What does that say about
   confidence-based cleaning?
3. Split by `created_at` month instead of customer. Is the score higher or lower
   than the customer split? Which split better matches deployment?
4. Add a `source` column with two values and compute per-source accuracy. Why
   can an aggregate score hide a broken source?
5. Extend the card with a "Maintenance" section describing who re-runs the
   generator and how the digest is verified in CI.

In [10]:
checks = {
    "messy data has planted defects": len(messy) > len(clean),
    "cleaning restores 600 unique rows": len(cleaned) == 600 and cleaned["text"].is_unique,
    "labels are normalised": set(cleaned["label"]) == {"billing", "technical", "account"},
    "random split leaks more than grouped": random_overlap > group_overlap,
    "group split keeps customers apart": set(group_train["customer_id"]).isdisjoint(set(group_test["customer_id"])),
    "noise hurts accuracy": noise_rows[-1]["test_accuracy"] < noise_rows[0]["test_accuracy"],
    "perfect agreement gives kappa 1": cohen_kappa(truth, truth) >= 0.999,
    "card has every section": not [section for section in required if section not in headings],
    "card written": card_path.exists(),
}
for name, passed in checks.items():
    print(f"{'ok' if passed else 'FAIL'} - {name}")
    assert passed, name
print("\nM05 validation passed.")

ok - messy data has planted defects
ok - cleaning restores 600 unique rows
ok - labels are normalised
ok - random split leaks more than grouped
ok - group split keeps customers apart
ok - noise hurts accuracy
ok - perfect agreement gives kappa 1
ok - card has every section
ok - card written

M05 validation passed.
